学習したモデルを実際に実行してみてどのような結果が帰ってくるのかを調べてみる

In [1]:
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd

/home/ayato-kaku/miniconda3/envs/CORY/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-09-11 10:56:40,904] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH


/home/ayato-kaku/miniconda3/envs/CORY/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: そのようなファイルやディレクトリはありません
collect2: error: ld returned 1 exit status


 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using untested triton version (2.2.0), only 1.0.0 is known to be compatible


In [8]:
POLICY_DIR    = "./outputs/cory_withIRM_last"   # CORY 側で save_pretrained した場所
IRM_MODEL_DIR = "../IRM/irm_iclr_model"         # IRM の学習済みモデル（重みが入ってるフォルダ）

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------- IRM のコードを import できるように --------
sys.path.append(os.path.abspath("../IRM"))  # irm_iclr.py があるディレクトリ
from irm_iclr import IRMScorer


In [13]:
# === CORY × IRM — Inference (安定版, OOM回避＆描画安全化, 事前チェックなし) ===

import os, sys
from typing import List, Dict

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead

# ---- パス設定（CORY/ で実行。IRM/ は兄弟ディレクトリ）----
POLICY_DIR    = "./outputs/cory_withIRM_last"   # CORY で save_pretrained した場所
IRM_MODEL_DIR = "../IRM/irm_iclr_model"         # IRM 学習済みモデルフォルダ

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- IRM のコード import（IRM/irm_iclr.py）----
sys.path.append(os.path.abspath("../IRM"))
from irm_iclr import IRMScorer

# ---- ポリシー&トークナイザ（半精度、省メモリ）----
tokenizer = AutoTokenizer.from_pretrained(POLICY_DIR)
# デコーダ専用モデルは左パディングにする
tokenizer.padding_side = "left"
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token
if getattr(tokenizer, "pad_token_id", None) is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.model_max_length = 512  # 入力は長くしすぎない

policy = AutoModelForCausalLMWithValueHead.from_pretrained(
    POLICY_DIR,
    torch_dtype=torch.float16 if DEVICE == "cuda" else None,
).to(DEVICE)
policy.eval()

# ---- IRM は CPUで（GPUメモリ温存）----
irm = IRMScorer(IRM_MODEL_DIR, max_length=512)
irm.model.to("cpu")
irm.device = "cpu"

# ---- 生成設定（控えめ）----
GEN_KWARGS = dict(
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=1.0,
    repetition_penalty=1.05,
    min_new_tokens=10,
    max_new_tokens=20,
    pad_token_id=tokenizer.pad_token_id,
)

@torch.no_grad()
def generate_texts(prompts: List[str], batch_size: int = 2, **gen_kwargs) -> List[str]:
    """
    小バッチで順次生成（GPUメモリ節約）。
    出力はプロンプト長ぶんをattention_maskで切り落として返す。
    """
    outs: List[str] = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i+batch_size]
        enc = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=384,   # 入力長を短め（さらに厳しいなら 256 などへ）
        ).to(DEVICE)
        gen = policy.generate(**enc, **gen_kwargs)
        for j in range(len(chunk)):
            in_len = int((enc["attention_mask"][j] > 0).sum().item())
            gen_ids = gen[j][in_len:]
            outs.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
        if DEVICE == "cuda":
            del enc, gen
            torch.cuda.empty_cache()
    return outs

MERGE_TEMPLATE = 'Please rewrite this to sound more positive while keeping meaning: "{}"'

def dual_role_round(reviews: List[str], gen_bs: int = 2) -> Dict[str, list]:
    """Observer → Pioneer の2段生成＋IRMスコア（IRMはCPU）"""
    # Observer
    obs = generate_texts(reviews, batch_size=gen_bs, **GEN_KWARGS)
    # Pioneer
    merged = [MERGE_TEMPLATE.format(q + r) for q, r in zip(reviews, obs)]
    pio = generate_texts(merged, batch_size=gen_bs, **GEN_KWARGS)
    # IRM Raw/Reward（CPU）
    scored_obs = irm.score_texts([q + r for q, r in zip(reviews, obs)])
    scored_pio = irm.score_texts([q + r for q, r in zip(reviews, pio)])
    return {
        "obs": obs,
        "pio": pio,
        "raw_obs": [float(s["raw_score"]) for s in scored_obs],
        "rew_obs":  [float(s["reward"])    for s in scored_obs],
        "raw_pio": [float(s["raw_score"]) for s in scored_pio],
        "rew_pio":  [float(s["reward"])    for s in scored_pio],
    }

# ---- 単発デモ ----
demo_text = "The movie was okay, but the pacing felt slow and the characters were underdeveloped."
out = dual_role_round([demo_text], gen_bs=1)

print("=== PROMPT ===")
print(demo_text)
print("\n=== OBS ===")
print(out["obs"][0][:600])
print("\n=== PIO ===")
print(out["pio"][0][:600])
print("\n[IRM] raw_obs=%.3f  raw_pio=%.3f" % (out["raw_obs"][0], out["raw_pio"][0]))
print("[IRM] rew_obs =%.3f  rew_pio =%.3f" % (out["rew_obs"][0], out["rew_pio"][0]))

# ---- バッチ評価（件数控えめ。さらに厳しければ 12→8 や gen_bs=1 に）----
ds = load_dataset("imdb", split="train").shuffle(seed=42).select(range(12))
reviews = [x["text"] for x in ds]
res = dual_role_round(reviews, gen_bs=2)

# ---- 可視化（NaN/Inf 除去 & ヒストが失敗したらECDFへ）----
def _to_finite_float_array(xs):
    arr = np.asarray([float(x) for x in xs], dtype=np.float32).reshape(-1)
    return arr[np.isfinite(arr)]

vals_raw = _to_finite_float_array(res["raw_obs"] + res["raw_pio"])
vals_rew = _to_finite_float_array(res["rew_obs"] + res["rew_pio"])

print(f"[debug] raw count={len(vals_raw)}, rew count={len(vals_rew)}")

def _safe_hist(data: np.ndarray, title: str, xlabel: str):
    if data.size == 0:
        print(f"[warn] {title} は空（全て NaN/Inf の可能性）")
        return
    try:
        plt.figure(figsize=(6,4))
        bins = int(max(5, min(20, max(3, data.size // 2))))
        plt.hist(data, bins=bins)
        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel("count")
        plt.show()
    except Exception as e:
        # ヒストが環境依存で失敗する場合のフォールバック（ECDF）
        print(f"[info] hist 失敗（{e}）。ECDFにフォールバックします。")
        xs = np.sort(data)
        ys = np.linspace(0, 1, xs.size, endpoint=True)
        plt.figure(figsize=(6,4))
        plt.plot(xs, ys)
        plt.title(title + " (ECDF)")
        plt.xlabel(xlabel)
        plt.ylabel("CDF")
        plt.show()

_safe_hist(vals_raw, "IRM Raw Score (Observer + Pioneer)", "raw score (~1..10)")
_safe_hist(vals_rew, "IRM Reward (Observer + Pioneer)", "reward (0..1)")

# ---- 先頭5件テーブル ----
def _safe5(xs):
    out = []
    for x in xs[:5]:
        try:
            v = float(x)
            if not np.isfinite(v):
                v = np.nan
        except Exception:
            v = np.nan
        out.append(v)
    return out

pd.DataFrame({
    "prompt": [r[:120] for r in reviews[:5]],
    "obs": [t[:120] for t in res["obs"][:5]],
    "pio": [t[:120] for t in res["pio"][:5]],
    "raw_obs": _safe5(res["raw_obs"]),
    "raw_pio": _safe5(res["raw_pio"]),
    "rew_obs": _safe5(res["rew_obs"]),
    "rew_pio": _safe5(res["rew_pio"]),
})


=== PROMPT ===
The movie was okay, but the pacing felt slow and the characters were underdeveloped.

=== OBS ===
 I also think that it just wasn't a good fit for this year's storyline."
I agree

=== PIO ===


And all in spite of having read one or two interviews on your recent work (of which you

[IRM] raw_obs=2.525  raw_pio=2.365
[IRM] rew_obs =0.169  rew_pio =0.152
[debug] raw count=24, rew count=24


ValueError: object __array__ method not producing an array

<Figure size 600x400 with 1 Axes>

ValueError: object __array__ method not producing an array

<Figure size 600x400 with 1 Axes>

TypeError: Cannot convert numpy.ndarray to numpy.ndarray